# **ICX360: объяснение ответа LLM через части контекста**

### Туториал к уроку «ICX360: Практика»

Neuronpedia показывает, что происходит **внутри** модели: какие признаки SAE активируются на токене, на какие токены признак реагирует сильнее всего, какие логиты он поднимает. Часто нужен другой ответ: на какую часть **входа** модель опиралась, когда выдала конкретный ответ. Если в контексте шесть предложений, а модель ответила по одному из них, то по какому? Объяснение ответа через части контекста называют in-context explainability, и ICX360 — открытая библиотека IBM Research с методами такого объяснения.

Что такое ICX360 (сверено 11.10.2026):

- репозиторий [github.com/IBM/ICX360](https://github.com/IBM/ICX360), лицензия Apache 2.0, [документация](https://ibm.github.io/ICX360/), статья о библиотеке: Wei et al., [ICX360: In-Context eXplainability 360 Toolkit](https://arxiv.org/abs/2511.10879), 2025;
- в библиотеке три метода:
  1. **MExGen** ([Multi-Level Explanations for Generative Language Models](https://arxiv.org/abs/2403.14459)) убирает части контекста, смотрит, насколько упала вероятность исходного ответа, и по этим падениям учит линейную модель-заменитель. Варианты C-LIME и L-SHAP — версии LIME и SHAP для генеративной модели;
  2. **CELL** ([CELL your Model: Contrastive Explanations for Large Language Models](https://arxiv.org/abs/2406.11785)) ищет правку промпта, после которой ответ меняется по заданной мере. Правки предлагает вторая модель-инфиллер (T5 или BART), поэтому метод тяжелее, и в этой тетради его нет;
  3. **Token Highlighter** ([Token Highlighter: Inspecting and Mitigating Jailbreak Prompts for Large Language Models](https://arxiv.org/abs/2412.18171)) — градиентный метод: важность фрагмента промпта равна норме градиента функции потерь заданного ответа по эмбеддингам токенов фрагмента. Авторы предложили его для поиска jailbreak-фрагментов, но целевой ответ можно задать любой.

План тетради: открытая модель Qwen2.5-0.5B-Instruct отвечает на вопрос по короткому тексту из шести предложений. Ее ответ объясняем тремя способами — C-LIME, L-SHAP и Token Highlighter — и проверяем объяснение вмешательством: убираем самое важное предложение и смотрим, что модель ответит теперь.

Тетрадь работает на CPU, GPU не нужен.

## Установка

ICX360 объявляет в своих требованиях Python ниже 3.13, `openai<2` и `datasets<4`, а в Colab сейчас Python 3.13, `openai` 2 и `datasets` 4. Методам из этой тетради ни `openai`, ни `datasets` не нужны, поэтому ставим пакет без зависимостей (`--no-deps --ignore-requires-python`) и с закрепленным коммитом репозитория, а две недостающие зависимости — отдельно. Остальное (`torch`, `spacy` с моделью `en_core_web_sm`, `scikit-learn`, `sentence-transformers`) в Colab уже есть; строка с `transformers` и spaCy-моделью нужна для запуска вне Colab.

In [ ]:
%pip install -q "transformers==5.18.0" "sentence-transformers==5.7.0" "en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
%pip install -q evaluate==0.4.6 toma==1.1.0
%pip install -q --no-deps --ignore-requires-python "icx360 @ git+https://github.com/IBM/ICX360.git@0c971ce8ef82b3228a3a34fcd0f184044ab943a0"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from icx360.algorithms.mexgen import CLIME, LSHAP
from icx360.algorithms.token_highlighter.th_llm import TokenHighlighter
from icx360.utils.model_wrappers import HFModel
from icx360.utils.general_utils import fix_seed, select_device

pd.set_option("display.max_colwidth", 100)
fix_seed(0)
device = select_device()
print(device)

## Модель и вопрос

Qwen2.5-0.5B-Instruct — instruct-модель на 0.5 млрд параметров с лицензией Apache 2.0. Версия весов закреплена коммитом на Hugging Face, чтобы ответы тетради не менялись при обновлении модели.

Одна строка ниже обязательна: `tokenizer.padding_side = "left"`. MExGen считает вероятность ответа сразу для пачки возмущенных контекстов разной длины, дополняет короткие паддингом и склеивает каждый контекст с токенами ответа, не передавая модели маску внимания (так устроен `ProbScalarizedModel` в закрепленной версии ICX360). У Qwen паддинг по умолчанию справа, и тогда паддинг-токены встают между контекстом и ответом: в нашем прогоне с правым паддингом удаление любого одного предложения роняло среднюю логарифмическую вероятность ответа на 0.9–1.5, и скоры всех шести предложений выходили от 0.52 до 1.37. С левым паддингом лишние токены стоят в начале и почти не мешают.

Контекст — шесть предложений об Эйфелевой башне. На вопрос отвечает одно из них, пятое, а понять его можно только вместе с четвертым: «this title» в пятом предложении — титул самого высокого сооружения из четвертого. Остальные предложения к вопросу не относятся.

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, revision="7ae557604adf67be50417f59c2c2f167def9a775")
tokenizer.padding_side = "left"   # см. абзац выше: без этой строки скоры MExGen искажены
model = AutoModelForCausalLM.from_pretrained(model_name, revision="7ae557604adf67be50417f59c2c2f167def9a775", dtype=torch.float32).to(device).eval()

sentences = [
    "The Eiffel Tower was built for the 1889 World's Fair in Paris.",
    "Its designer, Gustave Eiffel, also worked on the internal frame of the Statue of Liberty.",
    "The tower is 330 metres tall.",
    "For 41 years it was the tallest man-made structure in the world.",
    "In 1930 the Chrysler Building in New York took this title.",
    "Today the tower receives about seven million visitors a year.",
]
context = " ".join(sentences)
question = "Which building took the title of the tallest man-made structure from the Eiffel Tower?"

`HFModel` — обертка ICX360 над моделью Hugging Face: через нее объяснители генерируют текст и считают вероятность заданного ответа. Промпт передаем списком частей: тогда части-шаблоны (`"Context: "`, вопрос) можно пометить как неизменяемые, и объяснение будет только по контексту. Генерация жадная (`do_sample=False`), поэтому ответ при повторном запуске тот же.

In [ ]:
wrapped_model = HFModel(model, tokenizer)
model_params = {
    "max_new_tokens": 32,
    "do_sample": False,
    "chat_template": True,
    "system_prompt": "Answer the question in one short sentence using only the provided context.",
}

def make_prompt(ctx):
    return ["Context: ", ctx, "\n\nQuestion: ", question, "\n\nAnswer: "]

prompt = make_prompt(context)
output_orig = wrapped_model.generate("".join(prompt), text_only=False, **model_params)
print(output_orig.output_text[0])

**Задание 1.** Какое здание модель называет в ответе?

## MExGen C-LIME: какие предложения важны для ответа

C-LIME делает так:

1. делит контекст на предложения с помощью spaCy-модели `en_core_web_sm` (`segment_type="s"`);
2. строит возмущенные версии контекста: убирает каждое предложение по одному и каждую пару предложений (`max_units_replace=2` по умолчанию). Для шести предложений это $1 + 6 + 15 = 22$ версии вместе с исходной, и все они перебираются, а не сэмплируются;
3. для каждой версии считает скаляризатор `"prob"` — среднюю по токенам логарифмическую вероятность **исходного** ответа при этом контексте. Модель при этом ничего не генерирует заново;
4. учит линейную модель: признаки — какие предложения убраны, цель — средняя логарифмическая вероятность. Скор предложения — на сколько эта вероятность падает, когда предложение убирают. Положительный скор значит, что без предложения исходный ответ становится менее вероятным.

В `unit_types` метка `"p"` (paragraph) стоит у контекста, который делим на предложения, а метка `"n"` (not of interest) — у частей шаблона: их не убирают и скор им не дают.

In [ ]:
unit_types = ["n", "p", "n", "n", "n"]
ind_segment = [False, True, False, False, False]

fix_seed(0)
clime = CLIME(wrapped_model, scalarizer="prob", segmenter="en_core_web_sm")
out_clime = clime.explain_instance(prompt, unit_types, output_orig=output_orig, ind_segment=ind_segment,
                                   segment_type="s", model_params=model_params)

att = out_clime["attributions"]
clime_df = pd.DataFrame({"unit": att["units"], "type": att["unit_types"], "C-LIME": att["prob"]})
clime_df = clime_df[clime_df["type"] == "s"].reset_index(drop=True)
clime_df.round(3)

**Задание 2.** Какое предложение получило наибольший скор C-LIME? Во сколько раз его скор больше скора следующего по величине предложения? Ответ округлите до двух знаков.

In [ ]:
# отсортируйте clime_df по столбцу C-LIME и сравните два верхних скора
...

## MExGen L-SHAP

L-SHAP (local SHAP) оценивает значения Шепли предложений: вклад предложения усредняется по разным наборам других убранных предложений, а не берется из одной линейной модели. Для предложения L-SHAP рассматривает только ближайших соседей (`num_neighbors`), поэтому считает быстрее точного SHAP. Если C-LIME и L-SHAP согласны, объяснение не зависит от способа агрегации.

In [ ]:
fix_seed(0)
lshap = LSHAP(wrapped_model, scalarizer="prob", segmenter="en_core_web_sm")
out_lshap = lshap.explain_instance(prompt, unit_types, output_orig=output_orig, ind_segment=ind_segment,
                                   segment_type="s", model_params=model_params)

att = out_lshap["attributions"]
lshap_scores = [p for p, t in zip(att["prob"], att["unit_types"]) if t == "s"]
clime_df["L-SHAP"] = lshap_scores
clime_df.round(3)

**Задание 3.** Совпадает ли предложение с наибольшим скором у C-LIME и у L-SHAP? Какой у него скор L-SHAP (три знака)?

In [ ]:
# найдите idxmax по каждому столбцу
...

## Проверка вмешательством

Скор — утверждение о модели: «без пятого предложения исходный ответ становится менее вероятным, и сильнее, чем без любого другого». Его можно проверить напрямую. Уберем пятое предложение, посчитаем тем же скаляризатором среднюю логарифмическую вероятность исходного ответа и посмотрим, что модель сгенерирует теперь. Для сравнения сделаем то же с четвертым предложением, без которого пятое теряет смысл «this title».

In [ ]:
def without(i):
    return " ".join(s for j, s in enumerate(sentences) if j != i)

variants = {"весь контекст": context, "без 4-го предложения": without(3), "без 5-го предложения": without(4)}
for name, ctx in variants.items():
    logp = clime.scalarized_model.scalarize_output(inputs=["".join(make_prompt(ctx))], ref_output=output_orig,
                                                    **model_params)
    answer = wrapped_model.generate("".join(make_prompt(ctx)), **model_params)[0]
    print(f"{name}: средняя log-вероятность исходного ответа {logp.item():.3f}")
    print("   ответ:", answer)

**Задание 4.** Какое сооружение модель называет, когда из контекста убрано пятое предложение? В каком предложении контекста стоит это название?

## Многоуровневое объяснение: слова внутри главного предложения

Буква M в MExGen — multi-level. Предложение с наибольшим скором можно разбить на слова (`segment_type="w"`), а остальные оставить предложениями. Тогда объяснитель скажет, какие слова внутри главного предложения несут ответ, не тратя вызовы модели на слова неважных предложений.

In [ ]:
units = out_clime["attributions"]["units"]
types = out_clime["attributions"]["unit_types"]
scores = np.array(out_clime["attributions"]["prob"])
ind_words = np.zeros(len(units), dtype=bool)
ind_words[np.argmax(scores)] = True

fix_seed(0)
out_words = clime.explain_instance(units, types, output_orig=output_orig, ind_segment=ind_words,
                                   segment_type="w", model_params=model_params)
att = out_words["attributions"]
words_df = pd.DataFrame({"unit": att["units"], "type": att["unit_types"], "C-LIME": att["prob"]})
words_df = words_df[words_df["type"] == "w"].reset_index(drop=True)
words_df.round(3)

**Задание 5.** Какое слово пятого предложения получило наибольший скор C-LIME и какой у него скор (три знака)? Какой скор у слова «Building»?

In [ ]:
# отсортируйте words_df по скору и найдите строку со словом Building
...

## Token Highlighter: градиент вместо возмущений

MExGen вызывает модель десятки раз и смотрит только на выход — это метод черного ящика. Token Highlighter делает один прямой и один обратный проход: задает целевой ответ (аргумент `affirmation`), считает кросс-энтропию этого ответа при данном промпте и берет градиент по эмбеддингам токенов промпта. Скор фрагмента при `pooling="mean_norm"` — норма среднего градиента по токенам фрагмента.

В качестве целевого ответа зададим исходный ответ модели. Два отличия от MExGen: Token Highlighter строит промпт сам, через шаблон чата без системного промпта, и делит на предложения весь текст пользователя, поэтому «Context:» приклеивается к первому предложению, а вопрос получает свой скор.

In [ ]:
th = TokenHighlighter(model, tokenizer, segmenter="en_core_web_sm",
                      affirmation=output_orig.output_text[0], pooling="mean_norm")
user_text = "Context: " + context + "\n\nQuestion: " + question
out_th = th.explain_instance([user_text], ["p"], [True], "s")
th_df = pd.DataFrame({"unit": out_th["attributions"]["units"], "TH": out_th["attributions"]["scores"]})
th_df.round(4)

**Задание 6.** Какой фрагмент получил наибольший скор Token Highlighter? Во сколько раз его скор больше второго по величине (два знака)?

In [ ]:
# отсортируйте th_df по скору
...

## ICX360 рядом с Neuronpedia

- ICX360 отвечает на вопрос «из какой части входа взят этот ответ»: объяснение строится на уровне текста — предложений, фраз, слов — и одинаково работает с любой моделью, к которой есть доступ на генерацию и вероятности (MExGen) или градиенты (Token Highlighter).
- Neuronpedia отвечает на вопрос «что модель вычисляет внутри»: признаки SAE, их активации на токенах и вклады в логиты. Для этого нужны веса модели и обученный SAE к нужному слою.
- Вопросы складываются в цепочку: ICX360 находит фрагмент входа, от которого зависит ответ (здесь слово «Chrysler» в пятом предложении), а признаки SAE на токенах этого фрагмента показывают, что модель из него извлекает. Проверка вмешательством нужна на обоих уровнях: убрать фрагмент из входа, как в задании 4, или подавить признак, как в steering на Neuronpedia.
- Скоры ICX360 — свойства одного входа и одного ответа. Задание 4 показало, что даже согласные между собой методы не говорят, чем модель заменит убранный фрагмент: без пятого предложения она отвечает «Статуя Свободы», хотя скор второго предложения, откуда взято это название, мал, и это видно только после вмешательства.